## Introduzione

Questo notebook valuta quanto bene diversi modelli LLM (via Ollama) riescono a estrarre fatti strutturati da testi in italiano relativi a piani tariffari e condizioni contrattuali. Partendo da un insieme di chunk testuali e da una ground truth di fatti attesi, il notebook esegue l’estrazione, confronta i risultati e misura precisione, recall, errori di inventata, proprietà sbagliate, parsing falliti e consistenza tra più esecuzioni.

L’obiettivo è confrontare modelli diversi sullo stesso compito e capire quali combinano migliore accuratezza e stabilità nelle previsioni, senza dover scrivere codice di valutazione separato per ogni caso. È quindi utile per testare e documentare il comportamento degli LLM in un task di extraction fact-based.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pydantic import BaseModel
from ollama import chat

import csv

In [ ]:
GROUND: dict[str, dict] = {
    "chunk_1": {
        "text": "Il piano Premium include assistenza telefonica. Dal 1 gennaio 2025 il prezzo è di 49 euro mensili.",
        "facts": [
            {"predicate": "include", "args": ["piano_premium", "assistenza_telefonica"]},
            {"predicate": "prezzo", "args": ["piano_premium", "49", "euro_mese"]},
            {"predicate": "valido_da", "args": ["piano_premium", "2025-01-01"]},
        ],
    },
    "chunk_2": {
        "text": "Il piano Base non include assistenza telefonica, ma offre supporto via email.",
        "facts": [
            {"predicate": "non_include", "args": ["piano_base", "assistenza_telefonica"]},
            {"predicate": "include", "args": ["piano_base", "supporto_email"]},
        ],
    },
    "chunk_3": {
        "text": "A partire dal 1 gennaio 2025, il piano Premium costa 49 euro al mese.",  # parafrasi di chunk_1
        "facts": [
            {"predicate": "prezzo", "args": ["piano_premium", "49", "euro_mese"]},
            {"predicate": "valido_da", "args": ["piano_premium", "2025-01-01"]},
        ],
    },
    "chunk_4": {
        "text": "Dal 1 marzo 2025 il piano Premium passa a 59 euro mensili.",  # conflitto/aggiornamento rispetto a chunk_1
        "facts": [
            {"predicate": "prezzo", "args": ["piano_premium", "59", "euro_mese"]},
            {"predicate": "valido_da", "args": ["piano_premium", "2025-03-01"]},
        ],
    },
    "chunk_5": {
        "text": "Il piano Family consente fino a 6 utenti connessi contemporaneamente.",
        "facts": [
            {"predicate": "limite_utenti", "args": ["piano_family", "6"]},
        ],
    },
    "chunk_6": {
        "text": "I piani Base e Premium hanno una durata minima di contratto di 12 mesi.",
        "facts": [
            {"predicate": "durata_minima", "args": ["piano_base", "12_mesi"]},
            {"predicate": "durata_minima", "args": ["piano_premium", "12_mesi"]},
        ],
    },
}

MODELS = [
    # Famiglia Qwen (Piccolo vs Grande) - Context: 128k
    ("qwen", "qwen2.5:7b-instruct"), ("qwen", "qwen2.5:14b-instruct"),
    
    # Nuova Famiglia Llama da Hugging Face (Piccolo vs Grande) - Context: 128k
    ("llama-hf", "huggingface.co/QuantFactory/Llama-3.2-3B-Instruct-GGUF"), ("llama-hf", "huggingface.co/QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF"),

    # Famiglia Mistral (Piccolo vs Grande) - Context: 32k / 128k
    ("mistral", "mistral:7b-instruct"), ("mistral", "mistral-nemo:12b"),
]

RUNS = 3

## I tre prompt: evoluzione della strategia di estrazione

In questo notebook sono state confrontate tre versioni progressive di prompt per l’estrazione di fatti strutturati da testo in italiano. L'obiettivo comune è sempre lo stesso: leggere un testo naturale, identificare informazioni oggettive e trasformarle in una rappresentazione formale, compatta e facilmente confrontabile con una ground truth.

### Prompt 1: versione generica e flessibile
Il primo prompt è una configurazione semplice e generale. Dice al modello di:
- riconoscere fatti strutturati a partire da un testo;
- usare predicati in snake_case e minuscolo;
- normalizzare nomi ed entità;
- ignorare informazioni non oggettivamente verificabili.

Questa prima versione è utile perché impone un principio fondamentale: "non inventare". Inoltre introduce una regola di normalizzazione (es. "piano Premium" → "piano_premium"), che è importante per rendere i risultati comparabili tra testi diversi. Tuttavia, lascia il modello molto libero rispetto alla forma del risultato: non specifica una struttura JSON rigorosa, né dice in maniera precisa come i fatti devono essere rappresentati e organizzati.

Buon punto di partenza, ma la sua flessibilità può portare a:
- predicati variabili o poco coerenti;
- forme di output inconsistenti;
- casi in cui il modello raggruppa informazioni in modo poco standardizzato.

### Prompt 2: struttura esplicita e output più rigido
Il secondo prompt introduce un cambiamento importante: definisce esplicitamente la struttura del fatto come: `predicato(soggetto, valore)`

e specifica che:
- il primo argomento è sempre il soggetto;
- i fatti devono essere rappresentati come una lista JSON di oggetti con `predicate` e `args`;
- il prezzo deve essere separato in due parti distinte (`"30"` e `"euro_mese"`);
- non si devono unire informazioni diverse nello stesso fatto.

Questo prompt è migliore del precedente perché riduce l'ambiguità del formato di output. Invece di lasciare al modello una grande libertà stilistica, lo costringe verso un output standard, più facile da validare automaticamente. È inoltre più preciso sul modello semantico del dominio: distingue bene tra "soggetto", "valore", "unità di misura", "data di validità", ecc.

Il vantaggio principale è che rende l'estrazione più testabile e più legata a una rappresentazione machine-readable. Di conseguenza, i risultati possono essere confrontati in modo più affidabile con il ground truth e con altri modelli.

### Prompt 3: definizione ancora più forte con esempi e casi concreti
Il terzo prompt è la versione più robusta. Aggiunge un esempio esplicito per il caso delle date e chiarisce che la validità temporale va riportata con un fatto separato:
`valido_da`

Questa versione è migliore della seconda perché:
- rende ancora più evidente il principio di separazione delle informazioni;
- mostra concretamente come un fatto di tipo "prezzo" va espresso rispetto a uno di "data di validità";
- aiuta il modello a generalizzare meglio a scenari simili in cui un testo contiene più informazioni distinte legate allo stesso ente.

In altri parole, il terzo prompt non si limita a dire "usa una struttura rigorosa", ma mostra esempi realistici e casi borderline, aiutando il modello a capire non solo "come" rispondere, ma anche "quale livello di granularità usare". Questo riduce errori come:
- fusione di prezzo e data in un solo fatto;
- omissione della data di validità;
- rappresentazione delle entità in modo non normalizzato;
- produzione di output troppo liberi o inconsistenti.

### Perché questa progressione è utile
In sintesi, i tre prompt si differenziano per il grado di precisione e di vincolo imposto al modello:

- `PROMPT`: più generico, buona base, ma molto aperto;
- `PROMPT2`: più strutturato, definisce una rappresentazione formale chiara;
- `PROMPT3`: più dettagliato e guidato, include esempi che riducono il margine di errore.

L'evoluzione è quindi una progressione naturale verso un prompt "più forte": più specifico, più standardizzato e più in grado di far rispettare le regole di dominio. Questo è fondamentale in un task di fact extraction, dove il problema non è solo "capire il testo", ma anche "esprimere correttamente il risultato in una forma confrontabile e coerente".

L'idea generale è che, aumentando la chiarezza della semantica e della struttura del risultato, si riducono gli errori di parsing, gli output inconsistenti e le inferenze non supportate dal testo.

In [ ]:
PROMPT = """
    Sei un sistema che si occupa di estrarre fatti strutturati da un testo scritto in linguaggio naturale.

    Ciascun fatto ha un predicato (una parola, snake_case) e una lista di argomenti. 
    Esempi di predicati ammessi in questo dominio: include, non_include, prezzo, durata_minima, limite_utenti, valido_da.
    Puoi usarne altri se il testo lo richiede, ma resta coerente con lo stile (snake_case, minuscolo).

    Regole:
        - Non inventare informazioni assenti dal testo.
        - Riporta nomi ed entità in forma normalizzata: minuscolo, snake_case (es: "piano Premium" -> "piano_premium").
        - Se un'informazione non è chiaramente un fatto oggettivo, ignorale.

    Testo: "{text}"
"""

PROMPT2 = """
    Sei un sistema che si occupa di estrarre fatti strutturati da un testo scritto in linguaggio naturale.

    Ogni fatto ha ESATTAMENTE questa struttura: predicato(soggetto, valore).
    Il PRIMO argomento è SEMPRE il soggetto del fatto (l'entità di cui si parla), il resto sono i valori associati.

    Esempi di predicati ammessi: include, non_include, prezzo, durata_minima, limite_utenti, valido_da.

    Esempio corretto:
    Testo: "Il piano Gold include supporto prioritario."
    {{"facts": [{{"predicate": "include", "args": ["piano_gold", "supporto_prioritario"]}}]}}

    Esempio con prezzo (valore e unità separati, come due argomenti distinti):
    Testo: "Il piano Gold costa 30 euro al mese."
    {{"facts": [{{"predicate": "prezzo", "args": ["piano_gold", "30", "euro_mese"]}}]}}

    Regole:
    - Non inventare informazioni assenti dal testo.
    - Nomi ed entità sempre in minuscolo, snake_case (es. "piano Premium" -> "piano_premium").
    - Non unire più informazioni distinte (es. prezzo e data di validità) in un unico fatto: usa un fatto separato per ciascuna.
    - Se un'informazione non è chiaramente un fatto oggettivo, ignorala.

    Testo: "{text}"
"""

PROMPT3 = """
    Sei un sistema che si occupa di estrarre fatti strutturati da un testo scritto in linguaggio naturale.

    Ogni fatto ha ESATTAMENTE questa struttura: predicato(soggetto, valore).
    Il PRIMO argomento è SEMPRE il soggetto del fatto (l'entità di cui si parla), il resto sono i valori associati.

    Esempi di predicati ammessi: include, non_include, prezzo, durata_minima, limite_utenti, valido_da.

    Esempio corretto:
    Testo: "Il piano Gold include supporto prioritario."
    {{"facts": [{{"predicate": "include", "args": ["piano_gold", "supporto_prioritario"]}}]}}

    Esempio con prezzo (valore e unità separati, come due argomenti distinti):
    Testo: "Il piano Gold costa 30 euro al mese."
    {{"facts": [{{"predicate": "prezzo", "args": ["piano_gold", "30", "euro_mese"]}}]}}

    Esempio con data (sempre in formato ISO, YYYY-MM-DD):
    Testo: "Dal 15 marzo 2024 il piano Silver include il roaming."
    {{"facts": [{{"predicate": "include", "args": ["piano_silver", "roaming"]}}, {{"predicate": "valido_da", "args": ["piano_silver", "2024-03-15"]}}]}}

    Regole:
    - Non inventare informazioni assenti dal testo.
    - Nomi ed entità sempre in minuscolo, snake_case (es. "piano Premium" -> "piano_premium").
    - Non unire più informazioni distinte (es. prezzo e data di validità) in un unico fatto: usa un fatto separato per ciascuna.
    - Se un'informazione non è chiaramente un fatto oggettivo, ignorala.

    Testo: "{text}"
"""

## Spiegazione sintetica delle classi/strutture usate nel notebook

- ExtractedFact:
    - Rappresenta un singolo fatto estratto dal modello ha un 'predicate' (es. 'prezzo') e una lista di argomenti 'args' (es. ['piano_premium', '49', 'euro_mese'])
- ExtractionResult:
    - Contiene il risultato dell'estrazione per un testo: una lista di 'facts', cioè tutti i fatti riconosciuti dall'LLM
- FactTuple: 
    - Non è una classe vera e propria, ma un alias di tipo: serve a normalizzare un fatto come (predicato, (arg1, arg2, ...)). Questo rende più semplice confrontarlo con la ground truth
- EsitoChunk:
    - Memorizza la valutazione di un singolo chunk: quanti fatti sono corretti, mancanti, inventati, con proprietà errata, e se ci sono stati errori di parsing
- ResultModel:
    - Aggrega i risultati di un intero modello su tutti i chunk. Tiene conto di corretti, mancanti, inventati, proprietà errate, parsing falliti, precision, recall e consistenza media


In [ ]:
class ExtractedFact(BaseModel):
    predicate: str
    args: list[str]

class ExtractionResult(BaseModel):
    facts: list[ExtractedFact]

FactTuple = tuple[str, tuple[str, ...]]


@dataclass
class EsitoChunk:
    corretti: int = 0
    mancanti: int = 0
    inventati: int = 0
    proprieta_errata: int = 0
    parsing_fallito: bool = False
    estratti_norm: list[FactTuple] = field(default_factory=list)

@dataclass
class ResultModel:
    famiglia: str
    modello: str
    corretti: int = 0
    mancanti: int = 0
    inventati: int = 0
    proprieta_errate: int = 0
    parsing_falliti: int = 0
    totale_ground: int = 0
    total_extract: int = 0
    consistenza_media: float = 0.0

    @property
    def precision(self) -> float:
        return self.corretti / self.total_extract if self.total_extract else 0.0

    @property
    def recall(self) -> float:
        return self.corretti / self.totale_ground if self.totale_ground else 0.0


In [ ]:
def norm_fact(predicate: str, args: list[str]) -> FactTuple:
    return predicate.strip().lower(), tuple(a.strip().lower() for a in args)

## Main functions

- extract(model, text): Costruisce il prompt per il modello, chiama Ollama con schema JSON fissato (ExtractionResult) e restituisce il risultato parsato e un flag di successo del parsing
- evaluate(ground, estratti, parsing_ok): Confronta i fatti estratti con la ground truth, conta corretti, mancanti, inventati, proprietà errate e parsing falliti e normalizza i fatti in tuple (predicato, args...) per semplificare il confronto.
- compute_consistency(runs_facts): Misura la stabilità tra più run dello stesso modello. Usa Jaccard: intersezione / unione dei set di fatti. Se ci sono run valide, indica quanto i risultati sono coerenti tra loro.
- run_evaluate (ground, model): Questa funzione valuta tutti i modelli forniti su ogni chunk della ground truth. Per ciascun modello esegue un primo estrazione e altre `RUNS - 1` ripetizioni per verificare la stabilità delle predizioni. Per ogni chunk confronta i fatti estratti con quelli attesi, conta corretti, mancanti, inventati, proprietà errate e parsing falliti, e restituisce un `ResultModel` con metriche aggregate e consistenza media.

In [ ]:
def extract(model: str, text: str) -> tuple[ExtractionResult | None, bool]:
    prompt = pmt.format(text=text)
    try:
        response = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            format=ExtractionResult.model_json_schema(),
            options={"temperature": 0},
        )
        result = ExtractionResult.model_validate_json(response.message.content)
        return result, True
    except Exception as e:
        print(f"  [parsing failed: {e}]")
        return None, False


def evaluate(ground: list[dict], estratti: list[ExtractedFact] | None, parsing_ok: bool) -> EsitoChunk:
    esito = EsitoChunk(parsing_fallito=not parsing_ok)
    if not parsing_ok or estratti is None:
        esito.mancanti = len(ground)
        return esito

    ground_norm = [norm_fact(f["predicate"], f["args"]) for f in ground]
    estratti_norm = [norm_fact(f.predicate, f.args) for f in estratti]
    esito.estratti_norm = estratti_norm

    ground_rimasti = list(ground_norm)
    for pred, args in estratti_norm:
        if (pred, args) in ground_rimasti:
            ground_rimasti.remove((pred, args))
            esito.corretti += 1
            continue

        subject = args[0] if args else None
        match_subject = next(
            (g for g in ground_rimasti if g[0] == pred and g[1] and g[1][0] == subject),
            None,
        )
        if match_subject is not None:
            ground_rimasti.remove(match_subject)
            esito.proprieta_errata += 1
        else:
            esito.inventati += 1

    esito.mancanti = len(ground_rimasti)
    return esito


def compute_consistency(runs_facts: list[set[FactTuple]]) -> float:
    """Calcola l'indice Jaccard medio tra tutte le run valide."""
    valid_runs = [rf for rf in runs_facts if rf]
    if len(valid_runs) < 2:
        return 1.0 if len(valid_runs) == len(runs_facts) and len(runs_facts) > 0 else 0.0

    union = set().union(*valid_runs)
    if not union:
        return 1.0
    intersection = set.intersection(*valid_runs)
    return len(intersection) / len(union)


In [ ]:
def run_evaluate(ground: dict[str, dict], models: list[tuple[str, str]]) -> list[ResultModel]:
    results = []

    for family, model in models:
        print(f"\n--- Valutazione modello: {model} ({family}) ---")
        r = ResultModel(famiglia=family, modello=model)
        consistenze = []

        for chunk_id, dati in ground.items():
            print(f"  Elaborazione: {chunk_id}...")
            
            # Prima estrazione (utilizzata per evaluation e salvata per consistenza)
            first_result, first_ok = extract(model, dati["text"])
            esito = evaluate(dati["facts"], first_result.facts if first_result else None, first_ok)

            r.corretti += esito.corretti
            r.mancanti += esito.mancanti
            r.inventati += esito.inventati
            r.proprieta_errate += esito.proprieta_errata
            r.parsing_falliti += int(esito.parsing_fallito)
            r.totale_ground += len(dati["facts"])
            r.total_extract += len(esito.estratti_norm)

            # Raccolta run per consistenza riutilizzando la prima
            chunk_runs_facts: list[set[FactTuple]] = []
            if first_ok and first_result:
                chunk_runs_facts.append({norm_fact(f.predicate, f.args) for f in first_result.facts})
            else:
                chunk_runs_facts.append(set())

            # Esegue solo le RUNS rimanenti (RUNS - 1)
            for _ in range(max(0, RUNS - 1)):
                res, ok = extract(model, dati["text"])
                if ok and res:
                    chunk_runs_facts.append({norm_fact(f.predicate, f.args) for f in res.facts})
                else:
                    chunk_runs_facts.append(set())

            consistenze.append(compute_consistency(chunk_runs_facts))

        r.consistenza_media = sum(consistenze) / len(consistenze) if consistenze else 0.0
        results.append(r)

    return results


In [ ]:
def report(results: list[ResultModel]) -> None:
    print("\n" + "=" * 105)
    head = f"{'Modello':<24} {'Famiglia':<10} {'Prec.':>8} {'Recall':>8} {'Invent.':>8} {'Prop.err':>9} {'ParseX':>7} {'Consist.':>9}"
    print(head)
    print("-" * 105)
    for r in results:
        print(
            f"{r.modello:<24} {r.famiglia:<10} {r.precision:>8.2%} {r.recall:>8.2%} {r.inventati:>8} "
            f"{r.proprieta_errate:>9} {r.parsing_falliti:>7} {r.consistenza_media:>9.2%}"
        )
    print("=" * 105)


def save_csv(results: list[ResultModel], path: str = "./eval_report.csv") -> None:
    with open(path, "w", newline="", encoding="utf-8") as fp:
        writer = csv.writer(fp)
        writer.writerow([
            "modello", "famiglia", "precisione", "recall",
            "inventati", "proprieta_errate", "parsing_falliti", "consistenza_media"
        ])
        for r in results:
            writer.writerow([
                r.modello, r.famiglia, f"{r.precision:.3f}", f"{r.recall:.3f}",
                r.inventati, r.proprieta_errate, r.parsing_falliti, f"{r.consistenza_media:.3f}"
            ])
    print(f"Report salvato in {path}")


In [ ]:
results = run_evaluate(GROUND, MODELS)
report(results)
save_csv(results)